# ✍️ 01-数据结构与算法 · 主题 13：字符串（收尾篇）

> 字符串题四大类：**反转、匹配、回文、计数**——模板集中，性价比极高。
> 本 notebook 目标：**背熟双指针/哈希计数/KMP 概念 → 吃透 9 道高频题 → 通过 25 道自测**。

**前置**：主题 2（双指针）、主题 12（位运算）。**运行环境**：Python 3.8+。

## §1 字符串题四大类总览

| 类 | 代表题 | 核心工具 |
|----|--------|---------|
| 反转/变换 | 344 / 151 / 541 | 双指针、split、切片 |
| 匹配 | 28 / 459 | 朴素 / KMP / Rabin-Karp |
| 回文 | 125 / 680 / 5 | 双指针向中间、中心扩展 |
| 计数/哈希 | 242 / 49 / 387 | 计数器、排序键、ASCII 数组 |

**Python 字符串特点（常被问）**：

- **不可变**：`s[i] = 'x'` 会报错，必须构造新字符串或转 list
- 拼接 `+` 在循环中产生新对象 $O(n^2)$——用 `''.join(list)` 或 `list` 收集
- `s[::-1]` 反转、`s.split()` 按空白切、`s.split(',')` 指定分隔符

## §2 反转类：344 / 151

### 344 反转字符串（原地）

双指针从两端向中间交换：

```python
l, r = 0, len(s) - 1
while l < r:
    s[l], s[r] = s[r], s[l]
    l += 1; r -= 1
```
### 151 反转字符串中的单词

**思路**：`split()` 去掉多余空白 → 反转单词列表 → 用空格 join：

```python
return ' '.join(s.split()[::-1])
```

**复杂度**：$O(n)$ 时间（split 一次），空间 $O(n)$。

In [ ]:
def reverse_string(s):
    # 原地反转字符列表
    l, r = 0, len(s) - 1
    while l < r:
        s[l], s[r] = s[r], s[l]
        l += 1
        r -= 1
    return s

def reverse_words(s):
    # split 默认按任意空白切且去掉空串
    return ' '.join(s.split()[::-1])

print('344:', reverse_string(list('hello')))       # ['o','l','l','e','h']
print('344:', reverse_string(list('a')))           # ['a']
print('151:', reverse_words('the sky is blue'))    # 'blue is sky the'
print('151:', reverse_words('  hello world  '))    # 'world hello'
print('151:', reverse_words('a good   example'))   # 'example good a'

## §3 字符串匹配：28 找出字符串中第一个匹配项的下标

**朴素匹配**：

$$\text{匹配失败就 } i \gets i+1,\ j \gets 0 \quad \Rightarrow \quad O(n \cdot m)$$

**KMP 核心思想**：匹配失败时**不回退主串指针**，利用「前缀函数」跳到已匹配的前缀：

$$next[j] = \max\{ k < j : \text{模式串前 } k \text{ 个字符} = \text{后 } k \text{ 个字符} \}$$

**复杂度**：$O(n + m)$。

**Rabin-Karp（滚动哈希）概念**：

$$\text{hash}(s) = \sum s_i \cdot base^i \bmod p$$

窗口滑动时 $O(1)$ 更新哈希，冲突时再逐位验证。

> 💡 面试答「KMP 让主串不回退，复杂度 O(n+m)」即可；能写出 next 数组是加分。

In [ ]:
def str_str_naive(haystack, needle):
    if needle == '':
        return 0
    n, m = len(haystack), len(needle)
    for i in range(n - m + 1):
        if haystack[i:i + m] == needle:
            return i
    return -1

def str_str_kmp(haystack, needle):
    if needle == '':
        return 0
    # 计算 next 数组：next[i] = 前缀串最长相等前后缀长度
    m = len(needle)
    nxt = [0] * m
    j = 0
    for i in range(1, m):
        while j > 0 and needle[i] != needle[j]:
            j = nxt[j - 1]
        if needle[i] == needle[j]:
            j += 1
        nxt[i] = j
    # 匹配主串
    j = 0
    for i in range(len(haystack)):
        while j > 0 and haystack[i] != needle[j]:
            j = nxt[j - 1]
        if haystack[i] == needle[j]:
            j += 1
        if j == m:
            return i - m + 1
    return -1

print('朴素:', str_str_naive('sadbutsad', 'sad'))     # 0
print('朴素:', str_str_naive('leetcode', 'leeto'))    # -1
print('KMP:', str_str_kmp('hello', 'll'))             # 2
print('KMP:', str_str_kmp('ababcababc', 'abc'))       # 2
print('KMP 空模式:', str_str_kmp('a', ''))            # 0

## §4 回文类：125 / 680

### 125 验证回文串

**思路**：双指针从两端向中间，跳过非字母数字、忽略大小写：

```python
while l < r:
    while l < r and not s[l].isalnum(): l += 1
    while l < r and not s[r].isalnum(): r -= 1
    if s[l].lower() != s[r].lower(): return False
    l += 1; r -= 1
```

### 680 验证回文串 II（最多删一个字符）

**思路**：双指针遇到不等时，**只尝试删左或删右**（两种分支之一）：

$$\text{可行} = \text{is\_pal}(l+1, r) \lor \text{is\_pal}(l, r-1)$$

**复杂度**：$O(n)$，$O(1)$ 空间（每个分支最多再扫一遍）。

In [ ]:
def is_palindrome(s):
    l, r = 0, len(s) - 1
    while l < r:
        while l < r and not s[l].isalnum():
            l += 1
        while l < r and not s[r].isalnum():
            r -= 1
        if s[l].lower() != s[r].lower():
            return False
        l += 1
        r -= 1
    return True

def valid_palindrome_ii(s):
    def ok(l, r):
        while l < r:
            if s[l] != s[r]:
                return False
            l += 1
            r -= 1
        return True
    l, r = 0, len(s) - 1
    while l < r:
        if s[l] != s[r]:
            # 删左 or 删右
            return ok(l + 1, r) or ok(l, r - 1)
        l += 1
        r -= 1
    return True

print('125:', is_palindrome('A man, a plan, a canal: Panama'))  # True
print('125:', is_palindrome('race a car'))                     # False
print('125:', is_palindrome(' '))                              # True
print('680:', valid_palindrome_ii('abca'))                     # True（删 c）
print('680:', valid_palindrome_ii('abc'))                      # False
print('680:', valid_palindrome_ii('ab'))                       # True

## §5 计数/哈希类：242 / 49 / 387

### 242 有效的字母异位词

**两种写法**：

$$\text{Counter}(s) = \text{Counter}(t) \quad \text{或} \quad \text{sorted}(s) = \text{sorted}(t)$$

### 49 字母异位词分组

**分组键**：排序后的字符串（或字符计数元组）：

$$\text{key} = \text{sorted}(\text{word}) \Rightarrow \text{同字母异序词共享 key}$$

**复杂度**：$O(n \cdot k \log k)$（k 为单词平均长度）。

### 387 字符串中的第一个唯一字符

**思路**：Counter 统计 → 再扫一遍找第一个频次为 1 的：$O(n)$ 时间，$O(1)$ 空间（字母表固定 26）。

In [ ]:
from collections import Counter, defaultdict

def is_anagram(s, t):
    return Counter(s) == Counter(t)

def group_anagrams(strs):
    d = defaultdict(list)
    for w in strs:
        key = ''.join(sorted(w))    # 排序作分组键
        d[key].append(w)
    return list(d.values())

def first_uniq_char(s):
    cnt = Counter(s)
    for i, ch in enumerate(s):
        if cnt[ch] == 1:
            return i
    return -1

print('242:', is_anagram('anagram', 'nagaram'))   # True
print('242:', is_anagram('rat', 'car'))           # False
print('49:', group_anagrams(['eat', 'tea', 'tan', 'ate', 'nat', 'bat']))
print('387:', first_uniq_char('leetcode'))        # 0
print('387:', first_uniq_char('loveleetcode'))    # 2
print('387:', first_uniq_char('aabb'))            # -1

## §6 字符串转换：8 字符串转换整数（atoi）

**题目**：实现 `atoi`：忽略前导空格 → 处理正负号 → 读数字 → 越界截断到 $[2^{31}, 2^{31}-1]$。

**公式**：

$$\text{num} = \text{num} \times 10 + (ch - '0')$$

$$\text{clamp: } \begin{cases} -2^{31} & \text{num} < -2^{31} \\ 2^{31} - 1 & \text{num} > 2^{31} - 1 \end{cases}$$

**复杂度**：$O(n)$。

> ⚠️ 边界：`+` 号允许一次；`+-12` 非法返回 0；只有数字可读时部分读取。

In [ ]:
def my_atoi(s):
    i, n = 0, len(s)
    while i < n and s[i] == ' ':      # 1. 跳过空格
        i += 1
    sign = 1
    if i < n and (s[i] == '+' or s[i] == '-'):
        if s[i] == '-':
            sign = -1
        i += 1
    num = 0
    INT_MAX, INT_MIN = 2 ** 31 - 1, -2 ** 31
    while i < n and s[i].isdigit():   # 2. 读数字
        num = num * 10 + (ord(s[i]) - ord('0'))
        if sign * num >= INT_MAX:     # 3. 越界提前截断
            return INT_MAX
        if sign * num <= INT_MIN:
            return INT_MIN
        i += 1
    return sign * num

print('42:', my_atoi('42'))                     # 42
print('-42:', my_atoi('   -42'))                # -42
print('4193:', my_atoi('4193 with words'))      # 4193
print('0:', my_atoi('words and 987'))           # 0
print('max:', my_atoi('-91283472332'))          # -2147483648
print('0:', my_atoi('+-12'))                    # 0

## §7 常见 bug 与边界（面试踩坑区）

- **Python 字符串不可变**：原地修改必须转 list，改完 join 回来
- **循环拼接 O(n²)**：`s += ch` 在循环里每次都新建对象——用 `parts.append(ch)` + `''.join(parts)`
- **split 默认行为**：`'  a  b '.split()` → `['a','b']`（自动去空白去空串）；`split(' ')` 则保留空串——151 用前者
- **大小写**：125 用 `.lower()`；字母数字判断用 `.isalnum()`（含数字）
- **KMP next 数组下标**：`j = nxt[j - 1]` 循环必须写（否则死循环）
- **atoi 的越界判断时机**：先乘 10 再加就溢出，要在每次累加后立即判
- **空串/单字符**：`''` 模式返回 0；`'a'` 是回文、`'ab'` 删一个也是

## §8 面试追问与扩展（加分项）

### 8.1 KMP 为什么主串不回退（一句话）

匹配失败时，`next` 数组给出「模式串已匹配前缀中，与当前后缀相等的最长前缀」——从那里继续比，避免重复扫描主串。

### 8.2 最长回文子串（5）指路

DP / 中心扩展见主题 8 §4；面试优先中心扩展（$O(n^2)$ 常数小）。

### 8.3 字符串子串统计类

- 438 找到字符串中所有字母异位词（滑动窗口 + 计数比较）
- 3 无重复字符的最长子串（见主题 2）
- 76 最小覆盖子串（见主题 2）

### 8.4 编码题常考：压缩（443）/ 解码（394 见主题 4）

## §9 自测清单（L1 必会 / L2 进阶 / L3 挑战）

### L1 必会（10 题）

- [ ] 1. 反转字符串（LeetCode 344）
- [ ] 2. 反转字符串中的单词（151）
- [ ] 3. 验证回文串（125）
- [ ] 4. 有效的字母异位词（242）
- [ ] 5. 字符串中的第一个唯一字符（387）
- [ ] 6. 找出字符串中第一个匹配项的下标（28）
- [ ] 7. 最长公共前缀（14）
- [ ] 8. 字符串转换整数 atoi（8）
- [ ] 9. 重复的子字符串（459）
- [ ] 10. 整数反转（7）

### L2 进阶（10 题）

- [ ] 11. 字母异位词分组（49）
- [ ] 12. 验证回文串 II（680）
- [ ] 13. 找到字符串中所有字母异位词（438）
- [ ] 14. 字符串压缩（443）
- [ ] 15. Z 字形变换（6）
- [ ] 16. 简化路径（71，栈）
- [ ] 17. 单词规律（290，双向映射）
- [ ] 18. 字符串相乘（43，大数模拟）
- [ ] 19. 反转字符串中的单词 III（557）
- [ ] 20. 罗马数字转整数（13）

### L3 挑战（5 题）

- [ ] 21. 最短回文串（214，KMP 应用）
- [ ] 22. 编辑距离（72，见主题 8）
- [ ] 23. 通配符匹配（44，DP）
- [ ] 24. 最小覆盖子串（76，滑动窗口，见主题 2）
- [ ] 25. 解码方法（91，DP）

### 自测要点

- 5 分钟内盲写：344 + 125 + 242 + 28（朴素版）
- 能讲 KMP 的 next 数组含义与主串不回退原理
- 会写 atoi 的完整状态机（空格/符号/数字/越界）

## 附录：参考与结课寄语

- 《算法导论》第 32 章：字符串匹配（KMP/RK）
- LeetCode 分类题单：字符串

### 🎉 本系列完成：01-数据结构与算法 13 个主题全部收尾

**回看路线**：数组哈希 → 双指针滑动窗口 → 链表 → 栈队列 → 二分 → 二叉树 → 回溯 → 动态规划 → 贪心 → 图 → 堆 → 位运算 → 字符串。

**下一步**：`02-机器学习/`（线性回归/逻辑回归/SVM/树模型……）与 `03-深度学习/`（CNN/RNN/Transformer……）教学系列同样按此深度推进。

> 💡 本主题验收：`高频题单.md`「字符串」一节全部打勾；届时 01 系列 13 份 notebook + 题单即构成秋招算法完整兵器库。

## §10 高频补充：Rabin-Karp 滚动哈希 / 最长公共前缀 / 反转字符串中的单词

### Rabin-Karp 滚动哈希
- 把模式串哈希成整数 `hash = Σ s[i] * base^i mod mod`（base 取质数 31/131，mod 取 10**9+7）。
- 窗口滑动 O(1) 更新：`h = (h - s[l] * base^(k-1)) * base + s[r]`——减去最左字符、加入最右字符。
- 哈希相等只是候选，必须**逐字符确认**防冲突；期望复杂度 O(n+m)，最坏 O(n*m)。


In [ ]:
def rabin_karp(haystack, needle):
    # Rabin-Karp：滚动哈希在 haystack 中找 needle 首次出现下标，找不到返回 -1
    n, m = len(haystack), len(needle)
    if m == 0:
        return 0
    if m > n:
        return -1
    BASE, MOD = 131, 10 ** 9 + 7
    # 目标哈希 + 最高位权重 base^(m-1)
    target = 0
    for ch in needle:
        target = (target * BASE + ord(ch)) % MOD
    power = pow(BASE, m - 1, MOD)
    # 第一个窗口
    h = 0
    for ch in haystack[:m]:
        h = (h * BASE + ord(ch)) % MOD
    if h == target and haystack[:m] == needle:
        return 0
    for l in range(n - m):
        r = l + m
        # 窗口右滑：去掉最左字符（权重 power），加入最右字符
        h = ((h - ord(haystack[l]) * power) * BASE + ord(haystack[r])) % MOD
        if h == target and haystack[l + 1:r + 1] == needle:   # 命中后逐字符确认
            return l + 1
    return -1

print('RK:', rabin_karp('hello', 'll'), '| 内置 find:', 'hello'.find('ll'))          # 期望 2 / 2
print('RK:', rabin_karp('aaaaa', 'bba'), '| 内置 find:', 'aaaaa'.find('bba'))        # 期望 -1 / -1
print('RK 空模式:', rabin_karp('abc', ''))                                           # 期望 0
hay = 'a' * 10000 + 'b'
nd = 'a' * 5000 + 'b'
print('RK 长串:', rabin_karp(hay, nd), '| 内置 find:', hay.find(nd))                  # 期望 5000 / 5000

In [ ]:
def longest_common_prefix(strs):
    # 垂直扫描：逐列比较所有字符串同一下标的字符，遇到不同或越界即返回
    if not strs:
        return ''
    for i in range(len(strs[0])):
        ch = strs[0][i]
        for s in strs[1:]:
            if i >= len(s) or s[i] != ch:
                return strs[0][:i]
    return strs[0]

print('14:', longest_common_prefix(['flower', 'flow', 'flight']))  # 期望 fl
print('14:', repr(longest_common_prefix([])))                      # 期望 ''（空列表）
print('14:', longest_common_prefix(['a']))                         # 期望 a
print('14:', repr(longest_common_prefix(['dog', 'racecar', 'car'])))  # 期望 ''

In [ ]:
def reverse_words(s):
    # 省空间双反转版：整体反转 → 逐词反转 → write 指针原地压缩空格
    # 1) strip 去首尾空格，整体反转（单词顺序颠倒）
    # 2) 逐词 swap 反转（单词拼写恢复原序，此时顺序已正确）
    # 3) write 指针从左到右搬运，单词间只保留一个空格
    chars = list(s.strip())
    n = len(chars)

    def reverse_range(i, j):        # 反转 chars[i:j]（含 i，不含 j）
        while i < j - 1:
            chars[i], chars[j - 1] = chars[j - 1], chars[i]
            i += 1
            j -= 1

    reverse_range(0, n)             # 1) 整体反转
    i = 0
    while i < n:                    # 2) 逐词反转
        while i < n and chars[i] == ' ':
            i += 1
        j = i
        while j < n and chars[j] != ' ':
            j += 1
        reverse_range(i, j)
        i = j
    write = 0                       # 3) 压缩空格：单词间只留一个空格
    i = 0
    while i < n:
        if chars[i] != ' ':
            if write > 0:
                chars[write] = ' '
                write += 1
            while i < n and chars[i] != ' ':
                chars[write] = chars[i]
                write += 1
                i += 1
        else:
            i += 1
    return ''.join(chars[:write])

def reverse_words_split(s):
    # 直观版：手动 split + join（与双反转结果对照）
    return ' '.join(reversed(s.split()))

s1 = 'the sky is blue'
print('151:', repr(reverse_words(s1)), '| 对照:', repr(reverse_words_split(s1)))    # 期望 blue is sky the
s2 = '  hello   world  '
print('151:', repr(reverse_words(s2)), '| 对照:', repr(reverse_words_split(s2)))    # 期望 world hello
print('151:', repr(reverse_words('a good   example')))                             # 期望 example good a
print('151 全空格:', repr(reverse_words('   ')))                                   # 期望 ''